# Introduction
Event logs are the foundation of process mining. They capture records of activities within a system, providing information about when actions occur and what those actions are. For example, in GitHub Issue Events, actions such as assigning users, labeling issues, and closing issues are recorded. Together, these events tell the full story of the process from start to finish. Event logs can be transformed into differnt process graphs, which visually represent the flow of activities and how they connect. These graphs make it easier to identify inefficiencies, bottlenecks, and deviations from expected workflows. They provide valuable insights for process improvement and optimization. 

In this notebook, we demonstrate how to create process graphs using GitHub Issue Event Logs. Specifically, we use [Kaiaulu R package](https://github.com/sailuh/kaiaulu) to download and parse GitHub Issue Events. Capturing the lifecycle of an issue from creation, through assignment, discussion, and to closure.
Although these example's will focus on GitHub data, the techniques shown here can be applied to any event log that follows a similar format. This notebook serves as a demonstration of how event data can be prepared and explored for process mining. 

Note: For more information on process mining it is reccommeneded you refer to this [video](https://www.youtube.com/watch?v=XLHtvt36g6U&t=1181s). Sailuh/process mining relies heavily on the python package [pm4py](https://github.com/process-intelligence-solutions/pm4py), refer to the documentation for more detailed information regarding algorithms and models. 


### Requirements

-   Gihub Access Token 
-   [Kaiaulu](https://github.com/sailuh/kaiaulu) R package to download data via CLI
-   Python environment with [pm4py](https://github.com/process-intelligence-solutions/pm4py) installed
-   Faker installed for CSV data generation


In [9]:
# Python Imports

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import pandas
import pm4py
import subprocess

from api import *

#### Overview

To begin process mining, we first need the data in a properly formatted .csv file. This is where [Kaiaulu](https://github.com/sailuh/kaiaulu) comes in. Kaiaulu is a R package for software repository mining that provides a set of functions organized as an API. These functions allow us to downlaod and parse GitHub issue event data in our case. **In our examples we'll download and analysis issue event data from the Kaiaulu GitHub repository itself**. Rather than calling these functions from the project we will use an executable command-line interface (CLI) that is provided by kaiaulu to keep the processes indepenedent. Because this notebook and Kaiaulu are self contained, we assume the following folder organization: 

- ```kaiaulu/kaiaulu/exec/ghevents.R```
- ```kaiaulu/rawdata/```
- ```process_mining/notebooks/issue_event_processing.ipynb```

Note: In this structure, the outer kaiaulu/ folder represents the project scope, while the inner kaiaulu/ contains the actural R package with functions. It is assumed the process_mining/ folder and the outer kaiaulu/ folder are together in the same folder.



#### GitHub Events

The executable CLI requires a project configuration file to define the scope of the data we want to collect. Because we want to  download Kaiaulu GitHub data we will be using the correponding **kaiaulu.yml** config file. 
We are interested in the three fields from the config: 
- ```repo``` : the name of the GitHub repo
- ```owner``` : the GitHub user/organization
- ```issue_event``` : the save path for the raw issue event data (also default process graph save folder)

By default the issue_event path points to a directory called rawdata, located outside both the process_mining and Kaiaulu folders. The path structure is project dependent, hence the relative path is : **../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/**

Finally, the CLI requires a [GitHub API token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) to increase the request rate limite during data downloads. This token should be saved by default to the expected path **~/.ssh/github_token**

In [ ]:
cwd = os.getcwd()

kaiaulu_path = os.path.abspath(os.path.join("..", "..", "kaiaulu", "kaiaulu"))
token_path = "~/.ssh/github_token"

os.chdir(kaiaulu_path)
# To download use the parse command and specify the <config_path> <output_dir>.
command = f"Rscript exec/ghevents.R download conf/kaiaulu.yml --token_path={token_path}"
subprocess.run(command, shell=True, check=True)


Once we have the GitHub issue event rawdata (JSON) we may now parse it into a event log in .csv format. By default the .csv is saved to the folder where this package was downloaded. For example if it was downloaded to /Desktop/process_mining/... the .csv will have the following path /Desktop/issue_output.csv

In [ ]:
csv_folder = os.path.abspath(os.path.join(cwd, "..",".."))

# To parse use the parse command and specify the <config_path> <output_file>.
command = f"Rscript exec/ghevents.R parse conf/kaiaulu.yml {csv_folder}/issue_output.csv"
subprocess.run(command, shell=True, check=True)

os.chdir(cwd)

Now that we have a parsed event log in CSV format, we can load it using pandas and preview the first five rows to get a sense of the data structure.

In [ ]:
csv_path = os.path.join(csv_folder, "issue_output.csv")

df_issues = pd.read_csv(csv_path)
print(df_issues.head(5))

As we can see, the parsed event log conatins all the right data to see the entire process. However, just by reading each event induviually we can see it doens't give us good picture of the entire process. Process mining will solve this by stitching the events together into a visual. It provides a overview of how processes actualy unfold. 

It is important to note for process mining we will only be interested in the ```created_at```, ```event```, and ```issue_number``` columns. The others are not needed to generate graphs. 

#### Process Mining GitHub Events

Now we will generate the visualizations in practice using the data we dowloaded and parsed from the Kaiaulu Github issues. 

The process tree is our first visual we will generate. It uses the Inductive Miner Algorithm to generate a hierarchical visualization of the workflow. Capturing the sequences and dependecies of events. The algorithm is based on process discovery, and will highligh decision points and parallel activities. It is important for understanding the overall flow and structure of events and identifiying transitions and potential ineffcientices in the process. 

Note: You specifiy action as **view**, **save**, or **both** to determine what happens to the visualization. By default the generate functions will only view the graph. For our example we specify output_dir as the folder where process_mining was downloaded. This means for actions **save** and **both**, the images will be saved to the folder where process_mining was downloaded unless otherwise specified. 

In [ ]:
# Specify the directory to save images
output_dir = "../../"

generate_tree_inductive(csv_path, output_dir, action="both")

The visual is generated in the notebook and we can find the file located outside of process_mining.

We will create a BPMN (Buiness Process Model and Notation) with the same inductive miner from the last part creating the tree.
The BPMN model is a standard graphical representation of the process, it is the simpilest graph we will generate. It highlights the sequences of activies, decision points, and possible parallel paths in the process model. Compared to the tree it is easier to identify ineffcienceies and deviation the process flow from the simplistic nature. 

Filtering can also be applied to the process. By applying filtering we refine the model to focus on the most signifcant part of the process. It does this by reducing the noise and making the key patterns more apparent. This noise could be events that are not required to complete the process or that stray away from the main process for example. You can set the ```noise_threshold``` as a parameter, note the default is 0.0. We will set a 0.8 ```noise_threshold``` to reduce the graph size.

From here on we will only view the visuals to not create unncecesary files, but as stated above this parameter can be changed to **save** or **both**. You may need to expand the visual to see events.

In [ ]:
generate_graph_inductive(csv_path, noise_threshold=0.8)

Next the time edge weight graph visualizes the process flow with a focus on the time spent between activities. It uses the Performance DFG (Directly-Follows Graph). Every edge has a value which represents the interval between consecutive events. These time-based relations are useful to show where the process may need to be optimized to reduce the overall cycle time in the process. 

In [ ]:
generate_performance_graph_dfg(csv_path)

Now we are starting to get some complex grahps. This is because the overall process we are looking at (Kaiaulu Github Issue Events) is very large. It contains 300+ GitHub issues and a total 3600+ events at the creation of this notebook. At the end of this notebook provides graphs with minimal processes to understand the graphs better.

Another DFG (Directly-Follows Graph) generates the graph with occurrence edges. The weight is based on the frequency of transitions between nodes. This model highlights the most common paths, allowing for a better understanding of the domnant workflow patterns. It can help identify redudent steps in the process. 

In [ ]:
generate_count_graph_dfg(csv_path)

Finally, the Petri Net is a commonly used formal model used to represent workflows in automation and process mining. Unlike simplier models such as BPMN, Petri Nets represent conditions, tokens, and transitions. Instead of just showing the overview of the process. 

Important Symbols: 

- Circle with black center dot: Marks the start of the process.
- Circle with black square: Marks the end of the process.
- Empty circles: State or conditions in the process.
- Black boxes: Transitions that may be considered special. Example: Silent steps that don't correspond to events but may be needed for logical execution of the process.

In [ ]:
generate_petri_net_inductive(csv_path)

A useful function to call in conjunction when generating graphs is **start_end_activities(csv_path)**. It finds the start and end activites for the process. 
The start and end activies are the first and last recorded events in the process workflow. Marking the entry and exit points where no prior or further events occur. These activies are useful for reasoning through process grpahs and finding inefficencies in a non-graphical manner.

In [ ]:
start_end_activities(csv_path)

#### Understanding Process Graphs

Now that we have seen all the process mining functionally we can scale the process down to better understand what is happening. There will be three "experiements" we run.

For these we will need small event logs. These are artificially generated with functiuons from ```api/csv_generator.py```

In [ ]:
generated_csv_path = f"{csv_folder}/generated_csv.csv"

generate_fake_event_log(num_issues=1, num_events_per_issue=7, output_csv=generated_csv_path, seed=42)

event_log_df = pandas.read_csv(generated_csv_path)
print(event_log_df.head(7))

The event log is very straight forward  with each event appearing in order of ```created_at```. We will first generate a BPMN graph with no filtering to get a baseline.

In [ ]:
generate_graph_inductive(generated_csv_path, noise_threshold=0.0)

We will re-run the function to generate the graph. Notice that the resulting graph remains unchanged, demonstrating that the Inductive Miner algorithm produces consistent and reproducible results when applied to the same event log.

In [ ]:
generate_graph_inductive(generated_csv_path, noise_threshold=0.0)

Finally, we will change one event from the event log demonstrating the change to the process graph created. We are changing the event on row 4 to be mentioned instead of milestoned. This will shift the cycle in the process to the right changing it to "mentioned". 



In [ ]:
modify_event(event_log_df, row_index=4, new_event="mentioned")
print(event_log_df.head(7))


In [ ]:
event_log_df.to_csv("../../modified_csv.csv")

generate_graph_inductive("../../modified_csv.csv", noise_threshold=0.0)

#### Conclusion

Analyzing long and complex event logs can quickly become overwhelming, especially when trying to understand the underlying process structure. To manage this complexity, it is recommended to start with a smaller subset of data and gradually build up. A practical approach is to begin with only a few issues and examine how they translate into the process graph. This allows for clearer insights and easier debugging as the process grows.

This can be done by reading the csv into a pandas DataFrame and manipulating the data with Python. Alternatively the CSV can be modified with Excel or Google Sheets. The process graph generation functions remain flexible by requiring a CSV input, allowing them to be called from outside Python and integrated with other tools or systems.

By incrementally expanding the event log and re-generating the visualization, users can develop a deeper, more manageable understanding of the overall process behavior.